# AUTODATA 3.45 — extragere în cloud (varianta reparată)

**Corecție:** eroarea `OSError [Errno 107] Transport endpoint is not connected` este provocată de montarea Google Drive prin FUSE. Acest notebook nu mai folosește `drive.mount()` la transfer. În schimb, citește arhiva prin **Google Drive API în tranșe**, cu reîncercări și reluare din poziția locală existentă.

**Cum rulezi:** `Runtime → Run all`, apoi autorizezi contul Google care deține arhiva `Autodata__3.45.rar`. Nu trebuie să descarci nimic pe telefon.

- Citește arhiva originală din Drive fără a-i modifica accesul.
- Verifică integritatea (7z și MD5 oferit de Drive), apoi extrage în Colab.
- Creează folderul `Autodata_3_45_EXTRACTED` în **My Drive**, încarcă fișierele extrase cu reîncercări și publică `INVENTAR.json` și jurnalele.
- Nu execută programele din arhivă. Nu șterge originalul și nu distribuie public fișierele.

**Limite:** este nevoie de spațiu local Colab suficient pentru arhivă și conținutul dezarhivat și spațiu Drive pentru fișierele finale. Dacă sesiunea Colab este resetată, discul local temporar este pierdut. Dacă o celulă eșuează, cele de după ea nu trebuie executate; inspectează mesajul afișat. Operațiunile pe fișiere numeroase pot dura mult și sunt supuse limitelor Google.

In [ ]:
from google.colab import auth
# Autorizare Google: NU folosim drive.mount(), evitând FUSE și eroarea 107.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

credentials, project = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
print('Drive API autorizat. Nu este folosit drive.mount().')


In [ ]:
import os, re, json, sys, time, shutil, hashlib, subprocess, mimetypes, getpass
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from googleapiclient.errors import HttpError

DRIVE_FILE_ID = '1lC8pgAMU4RMkcatQClkzNnABJZpsKxZE'
ROOT = Path('/content/autodata_cloud_v2')
ROOT.mkdir(exist_ok=True)
ARCHIVE = ROOT / 'Autodata__3.45.rar'
EXTRACT_DIR = ROOT / 'files'
STATEFILE = ROOT / 'download_source.json'
OUT_NAME = 'Autodata_3_45_EXTRACTED'
CHUNK = 16 * 1024 * 1024
MAX_EXTRACTED_GIB = 20

info = api.files().get(fileId=DRIVE_FILE_ID,
                       fields='id,name,size,mimeType,md5Checksum,capabilities/canDownload',
                       supportsAllDrives=True).execute(num_retries=5)
if info.get('capabilities',{}).get('canDownload') is False:
    raise RuntimeError('Contul autorizat nu are drept de descărcare a arhivei.')
expected = int(info['size'])
assert info['name'] == 'Autodata__3.45.rar', 'ID-ul Drive indică alt fișier.'
print('Fișier identificat:', info['name'], '|', round(expected/2**30, 2), 'GiB')
print('Spațiu liber în Colab:', round(shutil.disk_usage('/content').free/2**30, 2), 'GiB')

# Dacă Google a modificat obiectul sursă după o copiere parțială, nu concatenăm două versiuni.
fingerprint = {'id': DRIVE_FILE_ID, 'size': expected, 'md5Checksum': info.get('md5Checksum')}
if ARCHIVE.exists() and STATEFILE.exists():
    if json.loads(STATEFILE.read_text()) != fingerprint:
        ARCHIVE.unlink()
elif ARCHIVE.exists() and not STATEFILE.exists():
    ARCHIVE.unlink()
STATEFILE.write_text(json.dumps(fingerprint))

if ARCHIVE.exists() and ARCHIVE.stat().st_size > expected:
    ARCHIVE.unlink()
offset = ARCHIVE.stat().st_size if ARCHIVE.exists() else 0
if shutil.disk_usage('/content').free < expected - offset + 150*1024*1024:
    raise RuntimeError('Spațiu insuficient în Colab pentru fișierul rămas de descărcat.')

session = AuthorizedSession(credentials)
url = f'https://www.googleapis.com/drive/v3/files/{DRIVE_FILE_ID}?alt=media'
print('Pornesc/continui transferul Drive API de la:', offset, 'octeți')
with ARCHIVE.open('ab') as dest:
    while offset < expected:
        end = min(expected - 1, offset + CHUNK - 1)
        last_error = None
        for attempt in range(1, 9):
            try:
                response = session.get(url,
                    headers={'Range': f'bytes={offset}-{end}', 'Accept-Encoding':'identity'},
                    timeout=(30, 180), stream=True)
                try:
                    if response.status_code != 206:
                        raise RuntimeError(f'Răspuns Google {response.status_code}: {response.text[:350]}')
                    cr = response.headers.get('Content-Range','')
                    m = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', cr)
                    if not m or (int(m[1]), int(m[2]), int(m[3])) != (offset,end,expected):
                        raise RuntimeError(f'Range incorect: {cr!r}')
                    block = response.content
                    if len(block) != end - offset + 1:
                        raise RuntimeError('Bloc primit incomplet')
                    dest.write(block)
                    dest.flush()
                    offset += len(block)
                    break
                finally:
                    response.close()
            except Exception as exc:
                last_error = exc
                print(f'Bloc {offset//CHUNK+1}: eroare {attempt}/8: {str(exc)[:140]}')
                if attempt != 8:
                    time.sleep(min(2**attempt, 45))
        else:
            raise RuntimeError(f'Descărcarea s-a întrerupt la {offset}/{expected}; '
                               'rulează DIN NOU această celulă în aceeași sesiune, fără restart. '
                               f'Ultima eroare: {last_error}')
        if offset == expected or offset % (CHUNK * 16) == 0:
            print(f'Transfer: {offset/expected:.1%} ({offset:,}/{expected:,} bytes)')

assert ARCHIVE.stat().st_size == expected
h_md5 = hashlib.md5()
h_sha = hashlib.sha256()
with ARCHIVE.open('rb') as f:
    while buf := f.read(8*1024*1024):
        h_md5.update(buf); h_sha.update(buf)
if info.get('md5Checksum') and h_md5.hexdigest().lower() != info['md5Checksum'].lower():
    raise RuntimeError('MD5 diferă de cel din Drive. Copia locală trebuie investigată.')
ARCHIVE_SHA256 = h_sha.hexdigest()
print('DESCĂRCARE VERIFICATĂ:', ARCHIVE_SHA256)


In [ ]:
if not shutil.which('7z'):
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','p7zip-full'], check=True)

def sevenzip(action, password=None):
    args=['7z',action,'-y']
    if password:
        args.append('-p'+password)
    if action=='x':
        args.append('-o'+str(EXTRACT_DIR))
    args.append(str(ARCHIVE))
    with (ROOT/f'{action}.log').open('w', encoding='utf-8') as log:
        result=subprocess.run(args, stdin=subprocess.DEVNULL, stdout=log,
                              stderr=subprocess.STDOUT, timeout=7200)
    return result.returncode

print('Testez integritatea RAR ...')
PASSWORD = None
valid=sevenzip('t')==0
if not valid:
    print((ROOT/'t.log').read_text(errors='replace')[-1500:])
    if input('Arhiva are parolă? da/nu: ').strip().lower() in ('da','d','yes','y'):
        PASSWORD=getpass.getpass('Parola arhivei: ')
        valid=sevenzip('t',PASSWORD)==0
if not valid:
    raise RuntimeError('Testul 7z a eșuat. Nu extrag fișiere deteriorate.')

args=['7z','l','-slt']
if PASSWORD: args.append('-p'+PASSWORD)
args.append(str(ARCHIVE))
listing=subprocess.run(args, capture_output=True,text=True, errors='replace',
                       timeout=1800, check=True).stdout
(ROOT/'listare_7z.txt').write_text(listing, encoding='utf-8')
entries = listing.split('----------',1)[-1]
expanded = sum(map(int,re.findall(r'(?m)^Size = (\d+)$',entries)))
paths = re.findall(r'(?m)^Path = (.*)$', entries)
for original_path in paths:
    name=original_path.replace('\\','/')
    part=PurePosixPath(name)
    if part.is_absolute() or '..' in part.parts or re.match(r'^[a-zA-Z]:',name):
        raise RuntimeError(f'Nume de fișier nesigur în arhivă: {original_path[:180]}')
print('Mărime estimată extragere:',round(expanded/2**30,2),'GiB')
if expanded > MAX_EXTRACTED_GIB*2**30:
    raise RuntimeError(f'Conținutul depășește plafonul de {MAX_EXTRACTED_GIB} GiB.')
if shutil.disk_usage('/content').free < expanded + 150*1024*1024:
    raise RuntimeError('Spațiu insuficient în Colab pentru extragere.')
EXTRACT_DIR.mkdir(exist_ok=True)
if sevenzip('x',PASSWORD)!=0:
    print((ROOT/'x.log').read_text(errors='replace')[-3000:])
    raise RuntimeError('7z nu a putut extrage arhiva.')
print('EXTRAGERE = OK')


In [ ]:
# Construim manifestul local înainte să începem încărcarea; acesta se va salva și în Drive.
records=[]
for root,dirs,names in os.walk(EXTRACT_DIR,followlinks=False):
    base=Path(root)
    dirs[:]=[d for d in dirs if not (base/d).is_symlink()]
    for name in names:
        p=base/name
        if p.is_symlink() or not p.is_file(): continue
        rel=p.relative_to(EXTRACT_DIR)
        records.append({'path':rel.as_posix(),'bytes':p.stat().st_size})
records.sort(key=lambda x:x['path'])
if not records: raise RuntimeError('Arhiva nu conține fișiere obișnuite.')
if len(records)>120000: raise RuntimeError('Peste 120000 fișiere; necesită procesare pe loturi.')
manifest={'archive':info['name'],'archive_drive_file_id':DRIVE_FILE_ID,
          'archive_bytes':expected,'archive_sha256':ARCHIVE_SHA256,
          'generated_utc':datetime.now(timezone.utc).isoformat(),
          'file_count':len(records),'extracted_bytes':sum(x['bytes'] for x in records),
          'files':records}
(ROOT/'INVENTAR.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
(ROOT/'INVENTAR.txt').write_text('\n'.join(f"{r['bytes']:>12} {r['path']}" for r in records),encoding='utf-8')
print(f'Inventar: {len(records)} fișiere, {manifest["extracted_bytes"]/2**30:.2f} GiB')
print('Categorii dominante:')
from collections import Counter
print(Counter(Path(r['path']).suffix.lower() or '[fără extensie]' for r in records).most_common(20))


In [ ]:
# Copiere prin Drive API, nu prin sistemul de fișiere montat.
# La rerulare: încarcă doar fișierele care lipsesc / au mărimi diferite.
DRIVE_FOLDER_MIME='application/vnd.google-apps.folder'

def escape_query(s):
    return s.replace('\\','\\\\').replace("'","\\'")

def list_children(folder_id):
    result={};token=None
    while True:
        page=api.files().list(q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken,files(id,name,mimeType,size)',pageSize=1000,
            pageToken=token, supportsAllDrives=True,includeItemsFromAllDrives=True
        ).execute(num_retries=5)
        for f in page.get('files',[]): result[f['name']]=f
        token=page.get('nextPageToken')
        if not token: break
    return result

def make_folder(name,parent_id,children_map):
    existing=children_map.get(name)
    if existing:
        if existing['mimeType']!=DRIVE_FOLDER_MIME:
            raise RuntimeError(f'Conflict de nume folder/fișier: {name}')
        return existing['id']
    created=api.files().create(body={'name':name,'mimeType':DRIVE_FOLDER_MIME,'parents':[parent_id]},
       fields='id,name,mimeType',supportsAllDrives=True).execute(num_retries=5)
    children_map[name]=created
    return created['id']

root_children=list_children('root')
output_id=make_folder(OUT_NAME,'root',root_children)
folder_id_map={'': output_id}
folder_contents={'': list_children(output_id)}

def folder_for(relpath):
    if not relpath or relpath=='.': return output_id,folder_contents['']
    if relpath in folder_id_map: return folder_id_map[relpath],folder_contents[relpath]
    parent=Path(relpath).parent.as_posix()
    if parent=='.':parent=''
    parent_id,parent_children=folder_for(parent)
    folder_id=make_folder(Path(relpath).name,parent_id,parent_children)
    folder_id_map[relpath]=folder_id
    folder_contents[relpath]=list_children(folder_id)
    return folder_id,folder_contents[relpath]

def upload_one(path,parent_id,children):
    name=path.name
    existing=children.get(name)
    if existing and existing.get('mimeType')!=DRIVE_FOLDER_MIME and        int(existing.get('size',-1))==path.stat().st_size:
        return 'skipped'
    if existing and existing['mimeType']==DRIVE_FOLDER_MIME:
        raise RuntimeError(f'Conflict nume: {name} este deja folder')
    media=MediaFileUpload(str(path),mimetype=mimetypes.guess_type(name)[0] or 'application/octet-stream',
                          resumable=True,chunksize=8*1024*1024)
    if existing:
        request=api.files().update(fileId=existing['id'],media_body=media,
            fields='id,name,size,mimeType',supportsAllDrives=True)
    else:
        request=api.files().create(body={'name':name,'parents':[parent_id]},
            media_body=media,fields='id,name,size,mimeType',supportsAllDrives=True)
    uploaded=None
    while uploaded is None:
        status,uploaded=request.next_chunk(num_retries=5)
    children[name]=uploaded
    return 'uploaded'

# Inventarul se publică primul, ca să putem analiza rezultatul inclusiv la upload parțial.
for index,file in enumerate([ROOT/'INVENTAR.json',ROOT/'INVENTAR.txt',
                              ROOT/'t.log',ROOT/'x.log',ROOT/'listare_7z.txt']):
    if file.exists():
        if file.name=='t.log': file_to_upload=file; name='verificare_7z.txt'
        elif file.name=='x.log': file_to_upload=file; name='extragere_7z.txt'
        else: file_to_upload=file;name=file.name
        # Nu redenumim originalul: copie locală pentru jurnal cu numele prietenos.
        if file.name in ('t.log','x.log'):
            friendly=ROOT/name
            shutil.copyfile(file,friendly)
            file_to_upload=friendly
        print('Inventar/jurnal:',file_to_upload.name,upload_one(file_to_upload,output_id,folder_contents['']))

uploaded=skipped=0
for i,rec in enumerate(records,1):
    p=Path(rec['path']); parent=p.parent.as_posix()
    if parent=='.':parent=''
    pid,existing=folder_for(parent)
    result=upload_one(EXTRACT_DIR/p,pid,existing)
    if result=='uploaded':uploaded+=1
    else:skipped+=1
    if i%50==0 or i==len(records):
        print(f'Drive: {i}/{len(records)} fișiere, noi/modificate={uploaded}, deja prezente={skipped}')
print('FINALIZAT: Folder Google Drive',OUT_NAME,'ID:',output_id)
print('Inventarul INVENTAR.json și fișierele extrase sunt disponibile în Google Drive.')
